# 🧠 24 · 前沿模型与推理时扩展

> 2024-2025 大模型热点：从「堆更大模型」转向「**推理时扩展（Test-Time Compute）**」——让模型在推理阶段想得更久、想得更对。

主线：前沿家族速览（LLaMA/Qwen/DeepSeek）→ 为什么推理时要扩展 → 采样-验证 → 过程奖励 → GRPO 训练 → 自测。

## 📋 本节要解决的问题

| # | 问题 | 答案 |
|---|---|---|
| 1 | 2024-25 主流模型长什么样？ | 大 MLP/稀疏 MoE/混合注意力 |
| 2 | 为什么不只堆参数？ | 推理时扩展的性价比曲线 |
| 3 | 多想几次能提对？ | Best-of-N / 多数投票 / pass@k |
| 4 | 谁来判断想得对不对？ | ORM vs PRM 奖励模型 |
| 5 | 怎么让模型学会长思考？ | R1：冷启动 + GRPO 强化学习 |
| 6 | GRPO 与 PPO 差在哪？ | 组内 advantage，省 Critic |

## 1️⃣ 前沿家族速览（一）：LLaMA 3

- **架构**：Decoder-only Transformer，SwiGLU FFN、RoPE、Grouped-Query Attention（GQA，8 个 KV head）、RMSNorm；
- **规模**：8B / 70B / 405B（2024），训练 15T+ token，128K 上下文；
- **数据**：高质量网页/代码/多语 + 蒸馏合成数据；后训练：SFT + RLHF（DPO 类）；
- 贡献：**开源生态事实标准**，后续模型（Qwen 等）对齐其 tokenizer/评测口径。

## 2️⃣ 前沿家族速览（二）：Qwen 与 DeepSeek

- **Qwen2.5/3**：中文最强开源族；Qwen3 引入**思考模式**（默认思考 + 非思考混合），MoE 版本（A3B：3B 激活/总 36B）；
- **DeepSeek-V3**：MoE 671B 总参/37B 激活，**DeepSeekMoE + 多 token 预测（MTP）**，MLA 注意力省 KV；训练成本极低（FP8、流水线）；
- **DeepSeek-R1**：首个开源「长思考」推理模型，靠 RL 内生 CoT（见 5️⃣）；
- 共性趋势：**稀疏化（MoE）+ 注意力减负（MLA/GQA）+ 长上下文 + 推理时扩展**。

## 3️⃣ 推理时扩展（Test-Time Compute）

训练预算固定后，**推理阶段的计算量也能买精度**：

- **宽度（width）**：并行采样多个答案再选（Best-of-N、self-consistency）；
- **深度（depth）**：更长的思维链、逐步验证、搜索回溯；
- **预算自适应**：简单题少想、难题多想（R1 的思考 token 分布可见）；
- 论文（Brown et al. 2024）结论：**自适应预算比固定预算更高效**——先预测难度再分配推理量。

In [1]:
import numpy as np, matplotlib.pyplot as plt
plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
np.random.seed(0)

# pass@k = 1-(1-p)^k：单次正确率 p，采样 k 次至少一次对
p = 0.40
ks = np.arange(1, 11)
passk = 1 - (1-p)**ks
plt.figure(figsize=(7.5, 3.2))
plt.plot(ks, passk, 'o-', lw=2)
plt.xlabel('采样次数 k（推理预算）'); plt.ylabel('pass@k 正确率')
plt.title('p=0.4：预算翻倍，正确率显著抬升')
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()
print('pass@1=%.2f → pass@5=%.3f → pass@10=%.3f' % (p, passk[4], passk[9]))

pass@1=0.40 → pass@5=0.922 → pass@10=0.994


C:\Users\24260\AppData\Local\Temp\ipykernel_40308\1564334363.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.grid(alpha=0.3); plt.tight_layout(); plt.show()


## 4️⃣ 采样 → 验证：「多想几次再交卷」

- **Best-of-N**：采样 N 个答案，用**奖励模型/验证器**挑最好的——只要验证器比生成准就赚；
- **Self-Consistency（自一致性）**：采样 N 条链，**多数投票**选出现最多的答案——不需要额外验证器；
- 代价：推理成本 ×N；收益随 N 递减（饱和）；
- 关键前提：**答案多样性**——温度要够，且任务有可投票的标准答案（数学/代码）。

In [2]:
def best_of_n(p, n, iters=3000):
    rng = np.random.RandomState(0)
    return np.mean([(rng.rand(n) < p).any() for _ in range(iters)])

def majority(p, n, iters=3000):
    rng = np.random.RandomState(0)
    return np.mean([(rng.rand(n) < p).sum() > n/2 for _ in range(iters)])

p = 0.60
print('单次采样正确率 p=%.2f' % p)
for n in (1, 3, 5, 9):
    print('  n=%d → Best-of-N %.3f | 多数投票 %.3f' % (n, best_of_n(p, n), majority(p, n)))
print('✓ 采样-验证策略显著提升：投票假设答案可分且互相关')

单次采样正确率 p=0.60
  n=1 → Best-of-N 0.596 | 多数投票 0.596


  n=3 → Best-of-N 0.937 | 多数投票 0.651
  n=5 → Best-of-N 0.991 | 多数投票 0.695


  n=9 → Best-of-N 1.000 | 多数投票 0.733
✓ 采样-验证策略显著提升：投票假设答案可分且互相关


## 5️⃣ 过程奖励（PRM）vs 结果奖励（ORM）

- **ORM（Outcome RM）**：只对最终答案打分——简单，但「过程错、答案碰对」无法区分；
- **PRM（Process RM）**：对**每一步**打分，定位哪步出错、指导搜索回溯；
- 数据：PRM 标注昂贵（人工逐步标注），可用**自动生成 + 弱监督**（如让模型自己证明每步）；
- 应用：**验证器 + 树搜索**——每步打分引导 MCTS 探索，推理时扩展的「深度」路线。

In [3]:
np.random.seed(2)
steps = 4
p_step = 0.9                                   # 每步独立正确率
trajs = np.random.rand(300, steps) < p_step   # 300 条轨迹
orm_acc = trajs.all(axis=1).mean()            # ORM：全步对才判对
prm_acc = (trajs.mean(axis=1) > 0.5).mean()   # PRM：多数步对
print('每步 p=%.2f，4 步任务' % p_step)
print('ORM 判定正确率 = %.3f（严格，一步错全错）' % orm_acc)
print('PRM 判定正确率 = %.3f（容忍单步错，能定位错步）' % prm_acc)
print('✓ PRM 信息更细：可告诉搜索「第 2 步出问题了」')

每步 p=0.90，4 步任务
ORM 判定正确率 = 0.637（严格，一步错全错）
PRM 判定正确率 = 0.920（容忍单步错，能定位错步）
✓ PRM 信息更细：可告诉搜索「第 2 步出问题了」


## 6️⃣ 搜索：把推理变成树搜索

验证器有了，就可以**搜索**：

- **MCTS**：节点=推理状态，子=下一步（生成 token/步骤），价值=PRM 分 + 蒙特卡洛回传；
- 选择 → 扩展 → 模拟 → 回传，重复至预算耗尽；
- 代表：AlphaGeometry（几何）、推理时 MCTS 增强版 o1 类系统、Agent 规划（任务分解 + 分支回退）；
- 工程上受限于：生成成本高、状态转移不可逆（token 级搜索粒度粗）→ 常在**步骤级**做搜索。

## 7️⃣ DeepSeek-R1：RL 炼出长思考

训练流程（公开版）：

1. **冷启动**：几千条人工 SFT（带长 CoT 的示例）教会模型「先想后答」；
2. **强化学习（GRPO）**：奖励 = **正确性**（答案比对）+ **格式**（用 `＜think＞...＜/think＞` 包住推理）；
3. RL 中涌现：自我反思、验证、语言混合规避（奖励驱动）——思考长度随难度自适应；
4. **蒸馏**：R1-Distill 把长思考能力蒸馏进小模型（Qwen/LLaMA 系列）。

对比 o1：o1 同样用 RL 训练隐式思维，但细节不公开；R1 的开源让「推理时扩展 + RL」变成可复现范式。

## 8️⃣ GRPO 公式（组内 advantage）

PPO 需要价值网络（Critic）估基线，GRPO 用**同问题的 G 个采样**做基线：

$$ A_i = \frac{r_i - \mathrm{mean}(r_1\dots r_G)}{\mathrm{std}(r_1\dots r_G) + \epsilon} $$

$$ \mathcal{L}_{GRPO} = -\mathbb{E}\left[\frac{1}{G}\sum_i \min\left(\frac{\pi_\theta}{\pi_{\text{old}}} A_i,\ \mathrm{clip}(\dots,1\pm\epsilon)A_i\right)\right] + \beta\, D_{KL}(\pi_\theta \| \pi_{\text{ref}}) $$

- 省掉 Critic → 显存与训练成本大降（V3/R1 训练成本核心来源）；
- KL 项约束策略不偏离参考模型太远；
- **只取组内相对**：整组都高分/低分时 advantage 仍可比（相对优劣信号）。

In [4]:
def grpo_advantage(rewards):
    r = np.array(rewards, float)
    return (r - r.mean())/(r.std() + 1e-6)

groups = {
    '组1 有差异': np.array([1.0, 0.8, 0.5, 0.2]),
    '组2 整体偏高': np.array([0.6, 0.5, 0.4, 0.3]),
    '组3 全部相同': np.array([0.5, 0.5, 0.5, 0.5]),
}
for name, rw in groups.items():
    a = grpo_advantage(rw)
    print('%-8s 奖励=%s → advantage=%s' % (name, rw, np.round(a, 3)))
print('✓ 组内相对化：组2 的 0.4 仍为负 advantage；组3 无信号(全 0，RL 不更新)')

组1 有差异   奖励=[1.  0.8 0.5 0.2] → advantage=[ 1.237  0.577 -0.412 -1.402]
组2 整体偏高  奖励=[0.6 0.5 0.4 0.3] → advantage=[ 1.342  0.447 -0.447 -1.342]
组3 全部相同  奖励=[0.5 0.5 0.5 0.5] → advantage=[0. 0. 0. 0.]
✓ 组内相对化：组2 的 0.4 仍为负 advantage；组3 无信号(全 0，RL 不更新)


In [5]:
# 温度与采样形状：温度越高分布越平，多样性↑（代价：质量下降）
def softmax(z):
    z = z - z.max()
    return np.exp(z)/np.exp(z).sum()

logits = np.array([2.0, 1.0, 0.5, 0.1])
for T in (0.2, 1.0, 2.0):
    print('T=%.1f → %s' % (T, np.round(softmax(logits/T), 3)))
print('✓ 低温=贪心近似，高温=均匀：推理时扩展常用 T≈0.6~1.0 采样')

T=0.2 → [0.993 0.007 0.001 0.   ]
T=1.0 → [0.575 0.211 0.128 0.086]
T=2.0 → [0.406 0.246 0.192 0.157]
✓ 低温=贪心近似，高温=均匀：推理时扩展常用 T≈0.6~1.0 采样


## 9️⃣ 推理时扩展的两种方向

| 方向 | 手段 | 代表 | 适用 |
|---|---|---|---|
| 宽度 | 并行采样 + 选择 | Best-of-N / 多数投票 | 有验证器/标准答案 |
| 深度 | 长思考 + 搜索 | R1/o1、MCTS、逐步验证 | 数学/代码/规划 |

- 现实系统**两者结合**：先采样多条，再对每条做验证/搜索；
- 成本结构差异：宽度吃**并行**（延迟不变）、深度吃**串行**（延迟线性涨）——实时场景权衡关键。

## 🔗 与相关话题连线

- **CoT**：提示层推理（few-shot 思维链）——推理时扩展的「最廉价形态」；
- **o1 系列**：隐式 RL 训练推理——推理时扩展的「工程化」；
- **蒸馏**：长思考模型 → 小模型（R1-Distill）——把推理时成本搬到训练时；
- **Agent**：工具调用 + 反思 + 搜索——推理时扩展的「外部化」；
- **RLHF vs GRPO**：PPO 用 Critic、GRPO 用组基线——成本差异与信号差异。

## 🎯 面试考点

1. LLaMA3/Qwen3/DeepSeek-V3 的架构关键词（GQA/MoE/MLA/MTP）；
2. 推理时扩展的定义与 width/depth 两方向；
3. pass@k 公式与 Best-of-N 的适用前提；
4. Self-Consistency 多数投票原理；
5. ORM vs PRM 的区别与标注成本；
6. R1 训练四步（冷启动/GRPO/涌现/蒸馏）；
7. GRPO advantage 公式与相对 PPO 的收益；
8. 温度/预算/延迟的工程权衡。

## ✅ 自测清单 + 衔接

- [ ] 手算 pass@k 并画预算-精度曲线；
- [ ] 手写 Best-of-N/多数投票蒙特卡洛验证；
- [ ] 手写 PRM vs ORM 判定对比；
- [ ] 手写 GRPO 组内 advantage（含无差异组）；
- [ ] 默写 GRPO 目标式（含 clip 与 KL 项）；
- [ ] 复述 8 条考点各 30 秒。

**衔接**：推理时扩展依赖高效推理（见 15-AI-Infra 05-KV Cache/06-vLLM/10-FlashAttention），也依赖 RL 训练（见 13-DPO 与 RLHF、21-PPO 与 GRPO 手推）。

In [6]:
# 自适应预算：简单题少想、难题多想（固定总预算更高效）
rng = np.random.RandomState(0)
difficulty = np.r_[rng.uniform(0, 0.5, 50), rng.uniform(0.5, 1, 50)]  # 精确 50 简单 + 50 难
rng.shuffle(difficulty)                    # 打乱顺序，仍是随机难度
p = np.where(difficulty < 0.5, 0.8, 0.3) # 简单/难题单次正确率
def passk(k): return 1 - (1-p)**k
fixed = passk(4).mean()                  # 固定预算：每任务 4 次 → 总 400
n_easy = int((difficulty < 0.5).sum()); n_hard = 100 - n_easy
k_s, B = 2, 400                            # 简单题 2 次；总预算 400 恒等
k_h = round((B - k_s * n_easy) / n_hard)   # 难题预算由总预算反推（≈6 次）
k_adapt = np.where(difficulty < 0.5, k_s, k_h)   # 自适应：简单少想、难题多想
assert k_adapt.sum() == 400                       # 同一总预算：2×50 + 6×50 = 400
print('固定 4 次/题: %.3f | 自适应 %d/%d 次: %.3f' % (fixed, k_s, k_h, passk(k_adapt).mean()))
print('✓ 同一预算，按难度分配推理量 → 精度更高（Brown et al. 2024）')


固定 4 次/题: 0.879 | 自适应 2/6 次: 0.921
✓ 同一预算，按难度分配推理量 → 精度更高（Brown et al. 2024）


## 🔟 o1 / 推理模型的细节

- **o1 风格**：RL 训练「先思考再回答」，思维链被内隐化（不强制显示）；
- 公开细节：大规模 RL + 过程奖励 + 搜索初始化；比 R1 的 GRPO 流程保密；
- 共同点：**推理时预算自适应**（简单题 token 少、难题 token 多）；
- 工程差异：o1 系 API 隐藏思考内容；开源系（R1/QwQ）显式 `think` 标签可观察可蒸馏。

In [7]:
# CoT 演示：把推理写成显式步骤（多步算术）
print('问题：一件商品八折后 64 元，原价多少？')
print('  1. 设原价 = x')
print('  2. 八折 → 0.8x = 64')
print('  3. 解得 x = 64/0.8 = 80')
print('  4. 结论：原价 80 元')
assert abs(64/0.8 - 80) < 1e-9
print('✓ CoT = 显式中间步骤：错误可定位、推理可审计（面试可演示）')

问题：一件商品八折后 64 元，原价多少？
  1. 设原价 = x
  2. 八折 → 0.8x = 64
  3. 解得 x = 64/0.8 = 80
  4. 结论：原价 80 元
✓ CoT = 显式中间步骤：错误可定位、推理可审计（面试可演示）


## 1️⃣1️⃣ RLHF vs GRPO：损失同源，advantage 不同

| 项 | PPO(RLHF) | GRPO |
|---|---|---|
| 基线 | Critic 价值网络（额外一个大模型） | 同问题 G 个采样的组内均值 |
| 显存 | Critic + 4 组网络 | 省 Critic |
| 信号 | 全局绝对 | 组内相对（组偏高也能分出优劣） |
| 适用 | 传统 RLHF | 开源推理模型（DeepSeek R1） |

clip 损失函数形式完全一致；**区别 100% 在 advantage 怎么来**。

In [8]:
def clip_loss(ratio, adv, eps=0.2):
    return -np.minimum(ratio*adv, np.clip(ratio, 1-eps, 1+eps)*adv).mean()

ratio = np.array([0.9, 1.1, 0.8, 1.2])
adv_ppo  = np.array([0.5, 0.3, -0.2, -0.6])          # Critic 基线
rew = np.array([1.0, 0.8, 0.5, 0.2])
adv_grpo = (rew - rew.mean())/(rew.std() + 1e-6)     # 组内标准化
print('PPO  adv =', np.round(adv_ppo, 3), '→ 损失 %.4f' % clip_loss(ratio, adv_ppo))
print('GRPO adv =', np.round(adv_grpo, 3), '→ 损失 %.4f' % clip_loss(ratio, adv_grpo))
print('✓ 损失式相同；GRPO 用组内相对信号替代 Critic')

PPO  adv = [ 0.5  0.3 -0.2 -0.6] → 损失 0.0250
GRPO adv = [ 1.237  0.577 -0.412 -1.402] → 损失 0.0660
✓ 损失式相同；GRPO 用组内相对信号替代 Critic


## 1️⃣2️⃣ 蒸馏：把推理能力装进小模型

- **SFT 蒸馏**：用 teacher 的（问题, 长 CoT 答案）对做监督微调（R1-Distill 主流）；
- **KL 蒸馏**：teacher 软标签（logits 分布）作为监督，保留类间相似度（温度 T>1）；
- 收益：推理时成本 ↓（小模型快），能力保留大部分；损失：深度推理/极端难题下降；
- 原则：蒸馏的是**行为**不是参数——数据质量 > 结构对齐。

In [9]:
def soft_ce(q_student, p_teacher, T=2.0):
    q = np.exp(np.array(q_student)/T); q = q/q.sum()
    p = np.exp(np.array(p_teacher)/T); p = p/p.sum()
    return -(p*np.log(q)).sum()

teacher = [3.0, 1.0, 0.2]              # teacher 认为「答案A」略优
student = [1.2, 0.9, 0.1]
print('teacher 分布(软标签):', np.round(np.exp(teacher)/np.exp(teacher).sum(), 3))
print('软标签 CE = %.4f（保留类间相似度信息）' % soft_ce(student, teacher))
print('✓ 蒸馏用软标签比硬标签信息更足（含「A 和 B 有多接近」）')

teacher 分布(软标签): [0.836 0.113 0.051]
软标签 CE = 1.0092（保留类间相似度信息）
✓ 蒸馏用软标签比硬标签信息更足（含「A 和 B 有多接近」）


## 1️⃣3️⃣ Agent：推理时扩展的外部化

- **工具调用**：把「想」外包给代码/检索/计算器（减少幻觉）；
- **反思循环**：生成 → 执行 → 报错 → 修正（ReAct、Self-Refine）；
- **多智能体**：规划/执行/审查分工，辩论制衡；
- 与推理时扩展的关系：预算不再只是 token 数，还包括**工具调用轮次与搜索深度**——同样要预算自适应。

## 🎯 进阶追问集

- 「自适应预算怎么实现？」→ 难度预测器 / RL 学停 / 验证器置信度触发继续思考；
- 「Best-of-N 与多数投票差别？」→ 前者要验证器挑最优，后者按频率投票（无需验证器但要求答案可投票）；
- 「GRPO 组内都高分怎么办？」→ 相对信号仍在（组内最好被推高），绝对值不重要；
- 「蒸馏会丢掉什么？」→ 深度搜索/极端难题的思考能力、长链推理的稳定性；
- 「Agent 的推理时扩展和 RL 关系？」→ 工具调用可建模为动作，可用同一套 GRPO 训练（R1 之后的主流方向）。

In [10]:
# 温度 × 采样次数：高温更随机 → 多次采样收益更大
def passk_sim(temp, k, trials=3000):
    rng = np.random.RandomState(int(temp*10))
    p_one = 0.6 if temp <= 0.3 else (0.5 - 0.2*(temp-0.3)/0.9)   # 高温单次更差
    return 1 - (1-p_one)**k

for T in (0.1, 0.5, 1.0):
    row = 'T=%.1f: ' % T
    for k in (1, 4, 16):
        row += 'k=%2d→%.3f  ' % (k, passk_sim(T, k))
    print(row)
print('✓ 低 T 单次准但多样少；高 T 多样但需多次采样——pass@k 衡量整个组合')

T=0.1: k= 1→0.600  k= 4→0.974  k=16→1.000  
T=0.5: k= 1→0.456  k= 4→0.912  k=16→1.000  
T=1.0: k= 1→0.344  k= 4→0.815  k=16→0.999  
✓ 低 T 单次准但多样少；高 T 多样但需多次采样——pass@k 衡量整个组合


## 1️⃣4️⃣ 推理时扩展的成本模型

$$ \text{精度}(\text{budget}) \uparrow,\quad \text{延迟/成本} = \text{budget} \times \text{单次代价} $$

- 决策线：**价值(正确率提升) vs 成本(算力×延迟)**——同预算下比命中率，同精度下比成本；
- 预算载体：token（CoT 长度）、采样次数 k、Agent 工具轮次、搜索深度；
- 工程实践：路由（简单题小模型直达、难题大模型多轮）+ 自适应早停；
- 面试口径：先定场景收益函数，再谈预算分配——比背论文数字更稳。

In [11]:
# 蒸馏温度扫描：T 越大软标签越平滑（保留相似度信息）
def soft_probs(logits, T):
    e = np.exp(np.array(logits)/T)
    return e/e.sum()

logits = [3.0, 1.0, 0.2]
for T in (0.5, 1.0, 2.0, 5.0):
    p = soft_probs(logits, T)
    print('T=%.1f → %s' % (T, np.round(p, 4)))
print('✓ T 大：第二/三类信息保留更多（A 与 B 相似度可被学生学到）')

T=0.5 → [0.9785 0.0179 0.0036]
T=1.0 → [0.836  0.1131 0.0508]
T=2.0 → [0.6194 0.2279 0.1527]
T=5.0 → [0.4461 0.299  0.2548]
✓ T 大：第二/三类信息保留更多（A 与 B 相似度可被学生学到）


## 🎤 前沿模型面试模拟问答（6 连）

Q「R1 和 o1 本质区别？」→ 方法公开（GRPO+RL+冷启动）vs 不公开；效果接近、成本开源路线更低；
Q「推理模型还依赖基座吗？」→ 是——冷启动 SFT + RL 都建在强基座/偏好数据上；
Q「CoT 越长一定越好？」→ 非线性：过短欠推、过长发散+成本爆炸——预算自适应必要；
Q「pass@k 什么不衡量？」→ 验证成本与选择策略（谁挑答案）；还需 self-consistency/vote；
Q「GRPO 省什么？」→ 省 Critic 网络与对应显存；代价是组内采样方差与并行开销；
Q「下一波？」→ 多模态推理（看图思考）、Agent 训练（工具+搜索）、测试时缩放政策学习（RL 学预算）。